In [2]:
%load_ext autoreload
%autoreload 2
%cd /opt/tiger/samantha

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/opt/tiger/samantha


In [5]:
from recipes.mi1.models.conditioners import T5Conditioner

In [144]:
cond_t5 = T5Conditioner(name="t5-base", n_embd=1024, finetune=False, device="cuda", word_dropout=0, normalize_text=False)
cond_t5.to("cuda")

sum([p.numel() for p in cond_t5.parameters()])

2023-12-19 01:23:27,883 - recipes.mi1.models.conditioners - INFO - [rank: 0] T5 will be evaluated with autocast as float32


787456

In [170]:
cond_t5.t5.config.task_specific_params

{'summarization': {'early_stopping': True,
  'length_penalty': 2.0,
  'max_length': 200,
  'min_length': 30,
  'no_repeat_ngram_size': 3,
  'num_beams': 4,
  'prefix': 'summarize: '},
 'translation_en_to_de': {'early_stopping': True,
  'max_length': 300,
  'num_beams': 4,
  'prefix': 'translate English to German: '},
 'translation_en_to_fr': {'early_stopping': True,
  'max_length': 300,
  'num_beams': 4,
  'prefix': 'translate English to French: '},
 'translation_en_to_ro': {'early_stopping': True,
  'max_length': 300,
  'num_beams': 4,
  'prefix': 'translate English to Romanian: '}}

In [183]:
outputs = cond_t5.tokenize(["disco all the way"])
outputs.input_ids

tensor([[5025,   32,   66,    8,  194,    1]], device='cuda:0')

In [174]:
result = cond_t5.forward(outputs)
print(result.embeds.shape)

torch.Size([1, 3, 1024])


In [162]:
result.embeds.shape

torch.Size([1, 3, 1024])

{'input_ids': tensor([[1006, 2684, 5025,   32, 2861,    7,    1]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [124]:
from recipes.mi1.models.tokenizers import MusicTagTokenizer

In [125]:
MusicTagTokenizer()

{'input_ids': tensor([[1006, 2684, 5025,   32, 2861,    7,    1]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [129]:
from recipes.mi1.models.conditioners import ArtistConditioner

music_artists = [
    "The Beatles",
    "Michael Jackson",
    "Madonna",
    "Elvis Presley",
    "Bob Dylan",
    "Ludwig van Beethoven",
    "Wolfgang Amadeus Mozart",
    "Beyoncé",
    "Lady Gaga",
    "Kanye West",
    "Taylor Swift",
    "Adele",
    "David Bowie",
    "Prince",
    "Freddie Mercury",
    "Whitney Houston",
    "Jay-Z",
    "Rihanna",
    "Kendrick Lamar",
    "Ed Sheeran"
]

n_bins = 32768
cond_artist = ArtistConditioner(n_bins=n_bins, cond_n_embd=128, n_embd=1024)

In [131]:
sum([p.numel() for p in cond_artist.parameters() if p.requires_grad])

4326400

In [113]:
token_ids, attention_mask = cond_artist.tokenize(music_artists)
result = cond_artist.forward(token_ids, attention_mask)
result.embeds.shape

torch.Size([20, 1, 1024])